In [ ]:
# Hücre 1: Ortam kurulumu
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/SentinelAI"
os.chdir(BASE)
print("Calisma dizini:", os.getcwd())

In [ ]:
# Hücre 2: Veriyi yükle + anomali skorlarını yeniden uygula
import pandas as pd
import joblib
import numpy as np

# 1. Veriyi yükle
df = pd.read_parquet(f"{BASE}/data/processed/logs.parquet")
print(f"Veri yuklendi: {len(df)} satir")

# 2. Kaydedilmiş modeli yükle
artifacts = joblib.load(f"{BASE}/models/isoforest.pkl")
model = artifacts["model"]
scaler = artifacts["scaler"]
features = artifacts["features"]
print(f"Model yuklendi. Ozellikler: {features}")

# 3. Anomali tahminini yeniden uygula
X = df[features].values
X_scaled = scaler.transform(X)   # fit_transform DEGIL, sadece transform!

df["anomaly"] = model.predict(X_scaled)          # -1 = anomali, +1 = normal
df["anomaly_score"] = -model.score_samples(X_scaled)  # yuksek = daha anormal

print(f"Anomali tahmini uygulandi: {(df['anomaly']==-1).sum()} anomali")

# 4. Sütunları da kaydet
df.to_parquet(f"{BASE}/data/processed/logs_with_anomaly.parquet", index=False)
print("Guncellenmis veri kaydedildi: logs_with_anomaly.parquet")

# 5. Kontrol
gerekli = ["attempts", "is_failed", "is_night", "ip_log_count",
           "user_failed_count", "anomaly", "anomaly_score", "true_label"]
eksik = [c for c in gerekli if c not in df.columns]
print(f"Eksik: {eksik}" if eksik else "Tum sutunlar mevcut")

In [ ]:
# Hücre 2.5: Doğru dosyadan okuduğumuzu garanti et
df = pd.read_parquet(f"{BASE}/data/processed/logs_with_anomaly.parquet")
print(f"Dogru dosya yuklendi: {len(df)} satir")
print(f"Sutunlar: {list(df.columns)}")

In [ ]:
# Hücre 4: Kural tabanlı sınıflandırma (Zaman Sızıntısı Düzeltildi)
def classify_threat(row):
    attempts   = row["attempts"]
    is_failed  = row["is_failed"]
    is_night   = row["is_night"]
    ip_count   = row["ip_log_count"]
    user_fail  = row["user_failed_count"]
    event      = row["event"]

    # --- Kural 1: Ağır brute-force ---
    if is_failed and attempts >= 20:
        return ("Brute Force", "T1110", f"Ayni oturumda {attempts} basarisiz deneme. Brute-force ile uyumlu.")

    # --- Kural 2: Orta seviye brute-force ---
    if is_failed and attempts >= 10:
        return ("Suspicious Failed Logins", "T1110.001", f"{attempts} basarisiz deneme. Password guessing olabilir.")

    # --- Kural 3: Gece başarılı giriş (ZAMAN SIZINTISI ONLEYICI) ---
    if is_night and event == "Login Success":
        # SADECE kullanicinin o andan ONCEKI (gecmis) loglarina bak
        gecmis_loglar = df[(df["user"] == row["user"]) & 
                           (df["is_night"] == 1) & 
                           (df["event"] == "Login Success") & 
                           (df["timestamp"] < row["timestamp"])]
        
        kullanici_gece_giris_sayisi = len(gecmis_loglar)

        if kullanici_gece_giris_sayisi < 20:
            return ("Suspicious Login Time", "T1078", f"Gece {int(row['hour']):02d}:00 civari giris. Kullanicinin gecmis davranisina uymuyor.")

    # --- Kural 4: Kullanıcıya yönelik yoğun fail ---
    if user_fail >= 20 and is_failed:
        return ("Targeted Credential Attack", "T1110.003", f"'{row['user']}' kullanicisina {user_fail} basarisiz giris.")

    # --- Kural 5: Anomali ama kurala uymuyor ---
    if row["anomaly"] == -1:
        return ("Unknown Anomaly", "TBD", "Model anomali isaretledi fakat bilinen bir kurala uymuyor.")

    # --- Kural 6: Normal ---
    return ("Normal", None, "Belirgin bir tehdit gostergesi yok.")


# Uygula
df[["threat_type", "mitre_id", "reason"]] = df.apply(
    lambda r: pd.Series(classify_threat(r)), axis=1
)

print("Siniflandirma tamamlandi\n")
print(df["threat_type"].value_counts().to_string())

In [ ]:
# Hücre 7: Model Değerlendirmesi
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score

df["pred_attack"] = (df["threat_type"] != "Normal").astype(int)
y_true = df["true_label"]
y_pred = df["pred_attack"]

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print("KARMASIKLIK MATRISI")
print(f"TN={tn}  FP={fp}")
print(f"FN={fn}  TP={tp}")
print()
print(f"Precision : {precision_score(y_true, y_pred):.3f}")
print(f"Recall    : {recall_score(y_true, y_pred):.3f}")
print(f"F1        : {f1_score(y_true, y_pred):.3f}")
print(f"FP orani  : {fp/(fp+tn):.3f}")
print(f"FN orani  : {fn/(fn+tp):.3f}")

In [ ]:
# Hücre 8: Sonuçları Kaydet
out_path = f"{BASE}/data/processed/logs_classified.parquet"
df.to_parquet(out_path, index=False)
print(f"Siniflandirilmis veri kaydedildi: {out_path}")